# Estimating the frequency-offset of a received signal

The purpose of this notebook is to compare different methods for estimating the frequency-offset of a received signal.

The basis for comparison will be the variance of the estimators and how this variance relates to the Cramer-Rao bound.

In [1]:
## Boilerplate instructions for importing NumPy and Matplotlib
# Import NumPy
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns
sns_context = "notebook"
sns.set_theme(context=sns_context, style="ticks")

# proper labeling of plots
from matplotlib.ticker import EngFormatter


### Cramer-Rao Bound

The Cramer-Rao bound is a theoretical lower bound on the variance of any unbiased estimator of a parameter; no (unbiased) estimator can have a variance lower than this bound.

For frequency estimation with a complex exponential signal (i.e., $x[n] = A e^{j(2 \pi f n + \phi)}$) of length $L_p$ and signal-to-noise ratio SNR (in dB), the Cramer-Rao bound is given by:

$$
\text{CRB}(f) = \frac{3}{2 \pi^2 L_p^3} \cdot \frac{1}{10^{\text{SNR\_dB} / 10}}
$$

In [2]:
def CRB_fd(L_p, SNR_dB):
    """Compute the Cramer-Rao Bound for frequency estimation.

    Args:
        L_p (int): Length of the observation window.
        SNR_dB (float): Signal-to-noise ratio in dB.

    Returns:
        float: The Cramer-Rao Bound for the given parameters.
    """
    return 3/(2*np.pi**2*L_p**3)*1/(10**(SNR_dB / 10))

## Base class for frequency estimators

The (abstract) base class below defines the interface for all frequency estimators. Any subclass should implement the `estimate` method, which takes a received signal as input and returns an estimate of its frequency offset.

**Note:** the `estimate()` method takes at its input the product of the received signal and the conjugate of the known preamble - not separate arguments for the received signal and the preamble.

In [3]:
class FrequencyEstimator:
    """Base class for frequency estimators.

    This class provides a common interface for all frequency estimator implementations.
    Subclasses should implement the `estimate` method.
    """
    def __init__(self):
        """Initialize the frequency estimator."""
        pass

    def estimate(self, signal):
        r"""Estimate the frequency offset of the given signal.

        Parameters
        ----------
        signal : np.ndarray
            The received signal for which to estimate the frequency offset. This signal is expected to be of the form $$r(t) = A e^{j (2 \pi \Delta f t + \phi)} + n(t)$$, where $A$ is the signal amplitude, $\phi$ is the signal phase, $\Delta f$ is the frequency offset, and $n(t)$ is additive noise.

            This signal is typically obtained by multiplying symbol-spaced samples of the received preamble with the conjugate complex of the known transmitted preamble.

        Returns
        -------
        float
            The estimated frequency offset.

        Raises
        ------
        NotImplementedError
            If the subclass has not implemented this method.
        """
        raise NotImplementedError("Subclasses must implement this method.")

### Frequency Estimator from Class

A simple frequency estimator that derives the frequency estimate from the phase difference between segments of the received signal was discussed in class. In other words, in this algorithm, we perform averaging over the signal *before* computing a phase difference.

The class below is a simplified (because we have already stripped of the preamble signal) version of the frequency estimator in `dc_632_26.estimator`.

In [4]:
class TwoHalvesEstimator(FrequencyEstimator):
    """Frequency estimator that splits the signal into `div`` segments and estimates the frequency offset based on the phase difference between the segments."""

    def __init__(self, div=2):
        """Initialize the two-halves frequency estimator.
        
        Parameters:
        ----------
        div : int, optional
            The number of segments to split the signal into for frequency estimation. Default is 2.
        """
        super().__init__()
        self.div = div

    def estimate(self, signal):
        r"""Estimate the frequency offset of the given signal by comparing the phase of the first and second `div` segments.

        Parameters
        ----------
        signal : np.ndarray
            The received signal for which to estimate the frequency offset. This signal is expected to be of the form $$r(t) = A e^{j (2 \pi \Delta f t + \phi)} + n(t)$$, where $A$ is the signal amplitude, $\phi$ is the signal phase, $\Delta f$ is the frequency offset, and $n(t)$ is additive noise.

        Returns
        -------
        float
            The estimated frequency offset.
        """
        symbols_per_segment = len(signal) // self.div
        freq_offset_estimates = []

        for segment_index in range(self.div - 1):
            start_index = segment_index * symbols_per_segment

            received_segment_a = signal[start_index : start_index + symbols_per_segment]
            received_segment_b = signal[start_index + symbols_per_segment : start_index + 2 * symbols_per_segment]

            segment_gain_a = np.sum(received_segment_a)
            segment_gain_b = np.sum(received_segment_b)

            phase_change = np.angle(segment_gain_b * np.conj(segment_gain_a))
            
            delta_f = phase_change / (2 * np.pi * symbols_per_segment)

            freq_offset_estimates.append(delta_f)

        return np.mean(freq_offset_estimates)
        

#### How good is this estimator?

We can evaluate the performance of this frequency estimator by comparing its variance to the Cramer-Rao bound. If the variance of the estimator is close to the Cramer-Rao bound, it indicates that the estimator is efficient and performs well.   

To measure the variance of the estimator, we can run multiple trials of the frequency estimation process and compute the sample variance of the resulting frequency estimates. This will give us an empirical measure of the estimator's performance, which we can then compare to the theoretical Cramer-Rao bound.

In [5]:
## Frequency Estimation Simulation
# Signal parameters
L_p = 32 # length of preamble
SNR_dB = 10  # Example value in dB

A = 1.
phi = np.random.uniform(0, 2 * np.pi)
delta_f = 0.1/L_p

N = 10000 # number of simulation trial
estimates = np.empty(N)

# the simulation loop
for i in range(N):
    # generate the noisy signal for this trial
    noise_power = A**2 / (10**(SNR_dB / 10))
    noise = np.sqrt(noise_power / 2) * (np.random.randn(L_p) + 1j * np.random.randn(L_p))
    t = np.arange(L_p)
    signal = A * np.exp(1j * (2 * np.pi * delta_f * t + phi)) + noise

    # run the frequency estimator for this trial and store the result
    estimator = TwoHalvesEstimator(div=2)
    estimates[i] = estimator.estimate(signal)

# summarize the simulation results
print(f"After {N} simulated frequency estimates: (SNR = {SNR_dB} dB, L_p = {L_p})")
print(f"Mean estimate: {np.mean(estimates):.5f}, True frequency: {delta_f} cycles/symbol")
print(f"Variance of estimates: {np.var(estimates):.5g}, Cramer-Rao Bound: {CRB_fd(L_p, SNR_dB):.5g}")
print(f"Mean squared error: {np.mean((estimates - delta_f)**2):.5g}, standard deviation: {np.std(estimates):.5g} cycles/symbol")


After 10000 simulated frequency estimates: (SNR = 10 dB, L_p = 32)
Mean estimate: 0.00313, True frequency: 0.003125 cycles/symbol
Variance of estimates: 6.096e-07, Cramer-Rao Bound: 4.6381e-07
Mean squared error: 6.0961e-07, standard deviation: 0.00078077 cycles/symbol


These are pretty good results; for this SNR and preamble length, the variance of the estimator is within 1.5x of the Cramer-Rao bound.

How does the performance change with different SNR values or preamble lengths? 

### Problem 1: Performance of "two-halves" algorithm

Conduct simulations for SNR varying between -5 dB and 20 dB and preamble length of 32, 64, and 128 symbols. Use at least 1000 trials for each combination.

Summarize your simulation results in a single plot that shows the variance (y-axis) versus SNR (x-axis) for the different preamble lengths. Make the y-axis logarithmic to better visualize the differences at low variance values. Use markers (e.g., `o-`) to clearly show each simulation point. Include a legend to distinguish between the different preamble lengths.

In [6]:
## Simulation code

In [7]:
## plot

#### Discussion of Results

### Problem 2: Implement and Analyze the Tretter/Kay algorithm

The paper "Feedforward Frequency Estimation for PSK: A Tutorial Review" by Morelli and Mengali describe  estimators due to Tretter and Kay, respectively. They show that the two algorithms are equivalent. 
The Tretter estimator is equation (19) in the paper and the Kay estimator is equation (26). You can use either for your implementation

Your task is to implement this algorithm (as a subclass of `FrequencyEstimator`). Then, conduct simulations and analysis similar to those for the "two-halves" algorithm in Problem 1.

In [8]:
## implement the Tretter/Kay estimator
class TretterKayEstimator(FrequencyEstimator):
    
    pass

In [9]:
## simulation code

In [10]:
## plot

#### Discussion of Results

### Problem 3: Implement and Analyze the Luise and Reggiani (L&R) Estimator

The same paper contains a description of the Luise and Reggiani (L&R) estimator; it is defined in equation (37).



In [11]:
## Implement the Luise and Reggiani (L&R) Estimator

class LuiseReggianiEstimator(FrequencyEstimator):
    
    pass

In [12]:
## simulation code

In [13]:
## plot

#### Discussion of Results

### Problem 4: Comparison of Frequency Estimators

Compare the three estimators (two-halves, Tretter/Kay, and Luise-Reggiani) in terms of their estimation accuracy. Also, assess the computational complexity of each algorithm.